# run_e17
Phase 2 Stage 2 CPU E17 worker: DEV-C off-diagonal scoring, source/target DEV-C calibration, and SCREEN evaluation (Colab CPU). Run cells 2, 3, and 4 in order. Configure only cell 3.

- Set `MODEL = 'qwen'` or `MODEL = 'gemma'` (one model per runtime; run both in parallel on separate CPU runtimes or sequentially).
- Requires certified Stage 1 `phase2_plan` and `phase2_e17_auth` (`phase2.e17_dev_c_transfer`).
- Verifies `en -> en` DEV-C diagonal reproduction (`atol=0, rtol=0`) before writing any output; keeps Phase 1 read-only, `CONFIRM` closed, and Stage 4 E19 cutoffs untouched.


In [ ]:
import os, sys, subprocess
from pathlib import Path
if sys.version_info[:2] not in ((3, 11), (3, 12)):
    raise RuntimeError('Use Python 3.11/3.12 with the pinned CPU requirements.')
for name in ('OPENBLAS_NUM_THREADS', 'OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[name] = '1'
from google.colab import drive
drive.mount('/content/drive')
REPO = Path('/content/Research-phase2-e17')
CODE_REF = 'phase2'  # Set to the approved Phase 2 branch or commit SHA to pin explicitly.
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Drazr/Research.git', str(REPO)], check=True)
if subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO, text=True).strip():
    raise RuntimeError('Use a clean checkout; preserve local work before continuing.')
subprocess.run(['git', 'fetch', 'origin', CODE_REF], cwd=REPO, check=True)
CODE_COMMIT = subprocess.check_output(['git', 'rev-parse', 'FETCH_HEAD'], cwd=REPO, text=True).strip()
subprocess.run(['git', 'checkout', '--detach', CODE_COMMIT], cwd=REPO, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO / 'requirements-stage2-cpu.txt')], check=True)
print('Fixed code commit:', CODE_COMMIT)
print('Restart if installation requests it, then rerun setup before continuing.')


In [ ]:
# Visible Stage 2 settings (edit only this cell).
PHASE1_ROOT = Path('/content/drive/MyDrive/phase1')
MODEL = 'qwen'                  # 'qwen' or 'gemma'
PLAN_REF = None                 # Required: {'path': ..., 'sha256': ...} or Path to phase2_plan _COMPLETE.json.
E17_AUTH_REF = None             # Optional if embedded in phase2_plan['e17_auth']; or {'path': ..., 'sha256': ...} / Path.
RESUME_DEV_C_REFS = None        # Optional {target_lang: {'path': ..., 'sha256': ...}} or Path to JSON mapping.
RESUME_CALIBRATORS_REF = None   # Optional {'path': ..., 'sha256': ...} or Path to phase2_e17_calibrators _COMPLETE.json.
RESUME_ANALYSIS_REF = None      # Optional {'path': ..., 'sha256': ...} or Path to phase2_analysis_block _COMPLETE.json.
OUTPUT_REF = None               # Optional mounted JSON path (e.g. PHASE1_ROOT / f'phase2/results/e17/{MODEL}_ref.json').


In [ ]:
import json, signal
if subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip() != CODE_COMMIT:
    raise RuntimeError('Checkout changed after setup.')
if subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO, text=True).strip():
    raise RuntimeError('Use a clean Phase 2 checkout.')
if MODEL not in ('qwen', 'gemma'):
    raise RuntimeError("Set MODEL to 'qwen' or 'gemma'.")
if not PHASE1_ROOT.is_dir():
    raise RuntimeError(f'Missing mounted PHASE1_ROOT directory: {PHASE1_ROOT}')
if PLAN_REF is None:
    raise RuntimeError('Set PLAN_REF to the certified Stage 1 phase2_plan reference before running.')
def _arg_for_ref(value, label):
    if isinstance(value, dict):
        rel_path = PHASE1_ROOT / value['path']
        if not rel_path.is_file():
            raise RuntimeError(f'Missing {label} manifest: {rel_path}')
        return json.dumps(value)
    ref_path = Path(value)
    if not ref_path.is_file():
        raise RuntimeError(f'Missing {label} manifest: {ref_path}')
    return str(ref_path)
command = [
    sys.executable, '-u', '-m', 'src.phase2.run_e17',
    '--phase1-root', str(PHASE1_ROOT),
    '--plan-ref', _arg_for_ref(PLAN_REF, 'phase2_plan'),
    '--model', MODEL,
    '--storage-scope', 'persistent-drive',
]
if E17_AUTH_REF is not None:
    command += ['--e17-auth-ref', _arg_for_ref(E17_AUTH_REF, 'phase2_e17_auth')]
if RESUME_DEV_C_REFS is not None:
    if isinstance(RESUME_DEV_C_REFS, dict):
        command += ['--resume-dev-c-refs', json.dumps(RESUME_DEV_C_REFS)]
    else:
        command += ['--resume-dev-c-refs', str(Path(RESUME_DEV_C_REFS))]
if RESUME_CALIBRATORS_REF is not None:
    command += ['--resume-calibrators-ref', _arg_for_ref(RESUME_CALIBRATORS_REF, 'resume calibrators')]
if RESUME_ANALYSIS_REF is not None:
    command += ['--resume-analysis-ref', _arg_for_ref(RESUME_ANALYSIS_REF, 'resume analysis')]
if OUTPUT_REF is not None:
    command += ['--output-ref', str(OUTPUT_REF)]
process = subprocess.Popen(
    command, cwd=REPO, env=os.environ.copy(),
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
blockers = []
try:
    for line in process.stdout:
        print(line, end='', flush=True)
        try:
            payload = json.loads(line)
            if isinstance(payload, dict) and payload.get('status') == 'blocked':
                blockers.extend(payload.get('blockers', []))
        except json.JSONDecodeError:
            pass
except KeyboardInterrupt:
    process.send_signal(signal.SIGINT)
    for line in process.stdout:
        print(line, end='', flush=True)
    process.wait()
    raise
code = process.wait()
if code != 0:
    detail = '; '.join(blockers) if blockers else f'exit code {code}'
    raise RuntimeError(f'E17 worker blocked/failed: {detail}')
